# Historical Realism Manual Verification

Use this notebook to verify V2 realism behavior on `convoy_layout_1` with both data and visuals/videos.

Scope:
- moving vs static U-boat behavior (same profile + seed)
- torpedo/observation/noise config visibility
- MP4/frames + timestamp snapshots
- simple pass/fail checks


In [1]:
# Imports
from __future__ import annotations

import importlib.util
import json
import shutil
from collections import Counter
from dataclasses import replace
from pathlib import Path

import numpy as np

from convoy_sim.attack_profiles import DEFAULT_ATTACK_PROFILE_LIBRARY
from convoy_sim.dynamics import ConvoyFormation, ConvoyKinematics, RouteLeg, RoutePlan, ZigZagPlan
from convoy_sim.feasibility import Environment
from convoy_sim.geometry import as_vec
from convoy_sim.realism import UBoatLeg, UBoatMotionPlan
from convoy_sim.viz_attack import render_attack_frame, save_attack_animation_mp4, save_attack_frames
from scenarios.convoy_profiles import get_convoy_layout_profile

try:
    from IPython.display import Video, display
except Exception:
    Video = None
    display = None


In [2]:
# --- Config ---
REPO = Path('/Users/matthewplambeck/Desktop/Convoy Layout Project')

# Keep notebook outputs near notebook files
OUTPUT_ROOT = REPO / 'notebooks/results/attack_manual_verification'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Verification target: convoy_layout_1 only
CONVOY_PROFILE = 'convoy_layout_1'
PROFILE_ID = 'P01'
SEED = 1942

SIM_DURATION_S = 180.0
FPS = 8
VIEW_BOUNDS = (-5000, 5000, -5000, 5000)

TRAIL_LENGTH_S = 600.0
TRAIL_LINEWIDTH = 0.20
TRAIL_ALPHA = 0.35

# Smaller marker to match normal plot feel
U_BOAT_MARKER_SIZE = 12.0

# Optional run parity references (kept at end)
BASELINE_RUN = REPO / 'results/runs/baseline/20260401_210338_baseline_test1'
RL_RUN = REPO / 'results/runs/rl/20260401_210544_rl_test1'

print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('CONVOY_PROFILE:', CONVOY_PROFILE)
print('PROFILE_ID:', PROFILE_ID)
print('SEED:', SEED)


OUTPUT_ROOT: /Users/matthewplambeck/Desktop/Convoy Layout Project/notebooks/results/attack_manual_verification
CONVOY_PROFILE: convoy_layout_1
PROFILE_ID: P01
SEED: 1942


In [3]:
# --- Build Scenario + Print Hard Data ---
import tomllib

has_matplotlib = importlib.util.find_spec('matplotlib') is not None
has_ffmpeg = shutil.which('ffmpeg') is not None
print('matplotlib:', has_matplotlib)
print('ffmpeg:', has_ffmpeg)

cfg = tomllib.loads((REPO / 'configs/baseline/default.toml').read_text(encoding='utf-8'))
sim_cfg = dict(cfg.get('simulation', {}))
noise_cfg = dict(sim_cfg.get('noise', {}))
env_cfg = dict(sim_cfg.get('environment', {}))

env = Environment(
    time_of_day=str(env_cfg.get('time_of_day', 'night')),
    visibility_m=float(env_cfg.get('visibility_m', 3500.0)),
    sea_state=int(env_cfg.get('sea_state', 4)),
    detection_risk_scale=float(env_cfg.get('detection_risk_scale', 1.0)),
)

layout_profile = get_convoy_layout_profile(CONVOY_PROFILE)
ships = layout_profile.build_ships()
class_counts = Counter([s.ship_class.value for s in ships])

profiles = {p.profile_id: p for p in DEFAULT_ATTACK_PROFILE_LIBRARY.profiles}
if PROFILE_ID not in profiles:
    raise ValueError(f'Unknown profile id: {PROFILE_ID}')

base_profile = profiles[PROFILE_ID]
moving_profile = replace(base_profile, u_boat_mode='moving')
static_profile = replace(base_profile, u_boat_mode='static')


def motion_plan_for(profile):
    legs = tuple(UBoatLeg(duration_s=float(a), heading_rad=float(b), speed_mps=float(c)) for a, b, c in profile.u_boat_motion_legs)
    return UBoatMotionPlan(
        initial_position=np.asarray(profile.u_pos, dtype=float),
        initial_heading_rad=float(profile.u_boat_initial_heading_rad),
        initial_speed_mps=float(profile.u_boat_initial_speed_mps),
        mode=str(profile.u_boat_mode),
        legs=legs,
        launch_time_s=float(profile.u_boat_launch_time_s),
        turn_rate_limit_rad_s=profile.u_boat_turn_rate_limit_rad_s,
        accel_limit_mps2=profile.u_boat_accel_limit_mps2,
    )


def scenario_data(profile):
    plan = motion_plan_for(profile)
    rng = np.random.default_rng(int(SEED))
    torpedoes = profile.build_torpedoes(rng, ships=ships, env=env)

    start_pos = plan.position_at(0.0)
    launch_pos = plan.position_at(float(profile.u_boat_launch_time_s))
    end_pos = plan.position_at(float(SIM_DURATION_S))

    return {
        'profile': profile,
        'plan': plan,
        'torpedoes': torpedoes,
        'start_pos': start_pos,
        'launch_pos': launch_pos,
        'end_pos': end_pos,
    }


moving = scenario_data(moving_profile)
static = scenario_data(static_profile)

print('=== Convoy Stats ===')
print('name:', layout_profile.name)
print('description:', layout_profile.description)
print('layout_kwargs:', layout_profile.layout_kwargs)
print('ship_count:', len(ships))
print('class_counts:', dict(class_counts))

print()
print('=== Simulation Realism Config (from baseline default.toml) ===')
print('environment:', env_cfg)
print('noise:', noise_cfg)


def print_case(label, data):
    p = data['profile']
    t0 = data['torpedoes'][0]
    print(f"\n=== {label} ===")
    print('profile_id:', p.profile_id)
    print('name:', p.name)
    print('attack_mode:', p.mode)
    print('u_boat_mode:', p.u_boat_mode)
    print('n_torpedoes:', p.n)
    print('torpedo_speed_mps:', p.speed)
    print('torpedo_max_run_time_s:', p.max_run_time)
    print('launch_delay_s:', p.launch_delay_s)
    print('salvo_interval_s:', p.salvo_interval_s)
    print('fan_base_bearing_rad:', p.base_bearing_rad)
    print('fan_spread_rad:', p.spread_rad)
    print('first_torpedo_heading_rad:', float(t0.heading_rad))
    print('first_torpedo_heading_deg:', float(np.degrees(t0.heading_rad)))
    print('u_boat_initial_heading_rad:', p.u_boat_initial_heading_rad)
    print('u_boat_initial_speed_mps:', p.u_boat_initial_speed_mps)
    print('u_boat_launch_time_s:', p.u_boat_launch_time_s)
    print('u_boat_turn_rate_limit_rad_s:', p.u_boat_turn_rate_limit_rad_s)
    print('u_boat_accel_limit_mps2:', p.u_boat_accel_limit_mps2)
    print('u_boat_motion_legs:', p.u_boat_motion_legs)
    print('start_pos:', data['start_pos'].tolist())
    print('launch_pos:', data['launch_pos'].tolist())
    print('end_pos:', data['end_pos'].tolist())
    print('dist_start_to_launch_m:', float(np.linalg.norm(data['launch_pos'] - data['start_pos'])))
    print('dist_start_to_end_m:', float(np.linalg.norm(data['end_pos'] - data['start_pos'])))


print_case('MOVING', moving)
print_case('STATIC', static)


matplotlib: True
ffmpeg: True
=== Convoy Stats ===
name: convoy_layout_1
description: Starting 7x6 convoy layout with all freighters.
layout_kwargs: {'n_rows': 6, 'n_cols': 7, 'spacing_along': 457.2, 'spacing_across': 1371.6, 'speed': 5.0, 'heading_rad': 0.0, 'length': 150.0, 'beam': 20.0, 'origin': array([0., 0.])}
ship_count: 42
class_counts: {'freighter': 42}

=== Simulation Realism Config (from baseline default.toml) ===
environment: {'time_of_day': 'night', 'visibility_m': 3500.0, 'sea_state': 4, 'detection_risk_scale': 1.0}
noise: {'sigma_heading_rad': 0.01, 'sigma_launch_delay': 0.05, 'sigma_speed_mps': 0.25, 'p_dud': 0.02}

=== MOVING ===
profile_id: P01
name: profile_01
attack_mode: fan
u_boat_mode: moving
n_torpedoes: 4
torpedo_speed_mps: 15.4333
torpedo_max_run_time_s: 486.0
launch_delay_s: 0.5
salvo_interval_s: 2.0
fan_base_bearing_rad: 4.04
fan_spread_rad: 0.0698
first_torpedo_heading_rad: -2.308281221109857
first_torpedo_heading_deg: -132.2547718988988
u_boat_initial_head

In [4]:
# --- Render Videos + Snapshot Figures ---
if not has_matplotlib:
    raise RuntimeError('matplotlib is required for visuals in this notebook.')


def render_case(label, data):
    case_dir = OUTPUT_ROOT / label.lower()
    case_dir.mkdir(parents=True, exist_ok=True)

    ships_local = get_convoy_layout_profile(CONVOY_PROFILE).build_ships()
    formation = ConvoyFormation(
        ships0=ships_local,
        convoy_origin0=as_vec(0.0, 0.0),
        convoy_heading0=0.0,
    )
    kin = ConvoyKinematics(
        route=RoutePlan(legs=[RouteLeg(duration_s=float(SIM_DURATION_S), heading_rad=0.0)]),
        zigzag=ZigZagPlan(enabled=False),
    )

    common = dict(
        color_by='class',
        show_trails=True,
        trail_length_s=float(TRAIL_LENGTH_S),
        trail_linewidth=float(TRAIL_LINEWIDTH),
        trail_alpha=float(TRAIL_ALPHA),
        show_footprint=False,
        ship_marker='ship',
        rotate_by_heading=True,
        use_hull_dimensions=True,
        trail_color='red',
        legend_bbox_to_anchor=(0.5, -0.20),
        view_bounds=VIEW_BOUNDS,
        hide_spines=True,
        figure_facecolor='lightgrey',
        show_u_boat=True,
        # Note: helper currently renders one fixed marker, not a moving marker per frame.
        u_boat_position=np.asarray(data['launch_pos'], dtype=float),
        u_boat_size=float(U_BOAT_MARKER_SIZE),
    )

    mp4_path = case_dir / f'{label.lower()}_{PROFILE_ID}.mp4'
    frames_dir = case_dir / 'frames'

    mp4_written = None
    try:
        save_attack_animation_mp4(
            str(mp4_path),
            ships_t0=ships_local,
            torpedoes=data['torpedoes'],
            t_start=0.0,
            t_end=float(SIM_DURATION_S),
            fps=int(FPS),
            dynamics=(formation, kin),
            **common,
        )
        mp4_written = mp4_path
    except Exception as exc:
        print(f'{label}: MP4 unavailable ({exc}); writing frames fallback...')
        save_attack_frames(
            str(frames_dir),
            ships_t0=ships_local,
            torpedoes=data['torpedoes'],
            t_start=0.0,
            t_end=float(SIM_DURATION_S),
            fps=int(FPS),
            dynamics=(formation, kin),
            **common,
        )

    # Snapshot frames for quick visual checks
    import matplotlib.pyplot as plt

    t_launch = float(data['profile'].u_boat_launch_time_s)
    times = [0.0, t_launch, float(SIM_DURATION_S)]
    names = ['t0', 'tlaunch', 'tend']
    for t_val, name in zip(times, names):
        fig, ax = plt.subplots(figsize=(7, 7), facecolor='lightgrey')
        render_attack_frame(
            ships_t0=ships_local,
            torpedoes=data['torpedoes'],
            t_global=float(t_val),
            t_max=float(SIM_DURATION_S),
            dynamics=(formation, kin),
            ax=ax,
            **common,
        )
        # Overlay start/end markers so U-boat movement is visually explicit.
        sp = np.asarray(data['start_pos'], dtype=float)
        ep = np.asarray(data['end_pos'], dtype=float)
        ax.scatter(sp[0], sp[1], s=16, c='#ffffff', edgecolors='#111111', linewidths=0.8, zorder=9, label='U-boat start')
        ax.scatter(ep[0], ep[1], s=16, c='#ffde59', edgecolors='#111111', linewidths=0.8, zorder=9, label='U-boat end')
        snap_path = case_dir / f'snapshot_{name}.png'
        fig.savefig(snap_path, dpi=150)
        plt.close(fig)

    return {
        'label': label,
        'mp4_path': None if mp4_written is None else str(mp4_written),
        'frames_dir': str(frames_dir) if frames_dir.exists() else None,
        'case_dir': str(case_dir),
    }


moving_out = render_case('moving', moving)
static_out = render_case('static', static)

print('moving outputs:', moving_out)
print('static outputs:', static_out)

if Video is not None and display is not None:
    if moving_out['mp4_path']:
        display(Video(moving_out['mp4_path'], embed=True))
    if static_out['mp4_path']:
        display(Video(static_out['mp4_path'], embed=True))


moving outputs: {'label': 'moving', 'mp4_path': '/Users/matthewplambeck/Desktop/Convoy Layout Project/notebooks/results/attack_manual_verification/moving/moving_P01.mp4', 'frames_dir': None, 'case_dir': '/Users/matthewplambeck/Desktop/Convoy Layout Project/notebooks/results/attack_manual_verification/moving'}
static outputs: {'label': 'static', 'mp4_path': '/Users/matthewplambeck/Desktop/Convoy Layout Project/notebooks/results/attack_manual_verification/static/static_P01.mp4', 'frames_dir': None, 'case_dir': '/Users/matthewplambeck/Desktop/Convoy Layout Project/notebooks/results/attack_manual_verification/static'}


In [5]:
# --- Pass/Fail Checks ---
move_dist = float(np.linalg.norm(moving['end_pos'] - moving['start_pos']))
static_dist = float(np.linalg.norm(static['end_pos'] - static['start_pos']))
launch_delta = float(np.linalg.norm(moving['launch_pos'] - static['launch_pos']))

print('move_dist_moving_m:', move_dist)
print('move_dist_static_m:', static_dist)
print('launch_pos_delta_moving_vs_static_m:', launch_delta)

assert move_dist > 0.0, 'FAIL: moving mode should have non-zero displacement.'
assert static_dist < 1e-9, 'FAIL: static mode displacement should be zero.'
assert launch_delta > 0.0 or moving_profile.u_boat_launch_time_s == 0.0, 'FAIL: expected different launch positions unless launch time is zero.'
assert len(moving['torpedoes']) > 0 and len(static['torpedoes']) > 0, 'FAIL: expected torpedoes in both modes.'

print('PASS: core historical realism checks succeeded for moving vs static U-boat.')


move_dist_moving_m: 360.0
move_dist_static_m: 0.0
launch_pos_delta_moving_vs_static_m: 0.0
PASS: core historical realism checks succeeded for moving vs static U-boat.


## Optional Run-Parity Check

This is not the core purpose of this notebook, but useful to quickly confirm canonical run metrics/manifests still match expected V2 values.


In [6]:
# --- Optional: Artifact + Metric Verification ---
def _read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

b_metrics = _read_json(BASELINE_RUN / 'metrics_summary.json')
r_metrics = _read_json(RL_RUN / 'metrics_summary.json')
b_manifest = _read_json(BASELINE_RUN / 'run_manifest.json')
r_manifest = _read_json(RL_RUN / 'run_manifest.json')

print('baseline static expected_hits:', b_metrics['static_baseline']['expected_hits'])
print('baseline heuristic expected_hits:', b_metrics['heuristic_baseline']['expected_hits'])
print('rl eval expected_hits:', r_metrics['evaluation']['expected_hits'])

assert b_manifest['realism']['u_boat_mode_default'] == 'moving'
assert r_manifest['realism']['u_boat_mode_default'] == 'moving'
assert b_manifest['seed_sets'] == r_manifest['seed_sets']
assert b_manifest['profile_splits'] == r_manifest['profile_splits']

print('PASS: run manifest realism + split/seed parity checks.')


baseline static expected_hits: 2.5225
baseline heuristic expected_hits: 2.495833333333333
rl eval expected_hits: 2.5225
PASS: run manifest realism + split/seed parity checks.
